# Evaluación del servicio de IA

Notebook de evaluación de la puntuación (1–5 estrellas) y la moderación de `ai-service`, con un conjunto de 200 reseñas etiquetadas a mano.

Contenido:
1. Carga y validación del dataset
2. Evaluación de la puntuación
3. Evaluación de la moderación y elección del umbral
4. Conclusiones

## 1. Carga y validación del dataset

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/reviews.csv")

El CSV se guardó con BOM y con las columnas alineadas con espacios, así que se lee con `utf-8-sig` y se limpian los espacios sobrantes.

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", skipinitialspace=True)
df.columns = df.columns.str.strip()
df["text"] = df["text"].str.strip()
df.head()

In [ ]:
print(f"Filas: {len(df)}")
print(f"Ids únicos: {df['id'].is_unique}")
print(f"Textos duplicados: {df['text'].duplicated().sum()}")
print(f"Textos vacíos: {(df['text'].str.len() == 0).sum()}")
print(f"Longitud mín/máx: {df['text'].str.len().min()} / {df['text'].str.len().max()}")

Etiquetas válidas: `stars` entre 1 y 5 (o vacía) y `flagged` igual a 0 o 1.

In [ ]:
assert df["flagged"].isin([0, 1]).all(), "flagged debe ser 0 o 1 en todas las filas"
assert df["stars"].dropna().isin([1, 2, 3, 4, 5]).all(), "stars debe estar entre 1 y 5"
df["stars"] = df["stars"].astype("Int64")

print("Distribución de flagged:")
print(df["flagged"].value_counts().sort_index().to_string())
print("\nDistribución de stars:")
print(df["stars"].value_counts(dropna=False).sort_index().to_string())

Cruce entre las dos etiquetas: las reseñas marcadas con `stars` son las que habría que comparar si no se hubieran bloqueado, y las que no tienen `stars` no se pueden puntuar.

In [ ]:
pd.crosstab(df["flagged"], df["stars"].isna().map({True: "sin stars", False: "con stars"}))